# CS4.406 A1 — MIND: lexical + semantic retrieval, ranking, Codabench submission

**What this notebook produces:** `prediction.zip` for
[MIND / Codabench 13967](https://www.codabench.org/competitions/13967/), plus every
artefact Q1–Q5 asks for (feature store, BM25 recall@K, embedding recall@K, evaluation
harness with slices + bootstrap CIs, leakage tests, ablation).

**How to run on Kaggle (free tier)**
1. Notebook → *Settings*: `Accelerator = GPU T4 x2` (only used to embed 130k headlines;
   the pipeline degrades gracefully to CPU), `Internet = On`,
   `Persistence = Files only` is not required.
2. *Add-ons → Secrets*: add `HF_TOKEN` with your HuggingFace token.
3. Run all. Expected wall-clock on a free T4 session: **~2.5–3.5 h**
   (download ~15 min, embeddings ~5 min, features+training ~35 min,
   test inference over 2.37 M impressions ~35 min, rest is analysis).
4. Download `/kaggle/working/prediction.zip` and upload it to Codabench.

**Design in one paragraph.** Candidate generation is *given* to us by the task (each
impression already ships its candidate list), so the leaderboard is won or lost in
*re-ranking*. We therefore build the retrieval machinery the assignment asks for
(BM25 inverted index, embedding ANN) and use it in two ways: (a) as candidate
generators measured by recall@K against the full catalogue — that is Q2/Q3, and
(b) distilled into cheap per-pair similarity features that feed a LightGBM ranker,
which is what actually scores the 88 M (impression, candidate) pairs of the test set.
Every article-level lookup is a numpy array indexed by an integer article code, every
user-level profile is one sparse-matrix product, and every per-pair similarity is one
blocked `einsum`. That is the only way 88 M pairs fit in a free Kaggle session.

## 0. Configuration

Everything tunable lives here. The sample sizes are chosen so that a full run fits
comfortably in a 12 h / 30 GB Kaggle session with ~3× headroom; raise `TRAIN_IMPRESSIONS`
if you have quota to spare (returns diminish quickly past ~1 M impressions).

In [15]:
import gc
import json
import warnings
import os
import time
import zipfile
from pathlib import Path

import numpy as np
import polars as pl
import scipy.sparse as sp

warnings.filterwarnings("ignore", category=DeprecationWarning)
TEST_MODE = os.environ.get("A1_TEST_MODE") == "1"  # tiny synthetic run used for CI

CFG = dict(
    seed=42,
    # ---- sampling -------------------------------------------------------
    train_impressions=450_000,     # sampled from MINDlarge_train (~2.23 M)
    es_impressions=120_000,        # early-stopping split (last day of train, temporal)
    eval_impressions=80_000,       # held-out MINDlarge_dev — the honest offline number
    recall_queries=3_000,          # users used for the recall@K study (Q2/Q3)
    # ---- representations ------------------------------------------------
    hist_len=30,                   # long-term profile: last N clicks
    hist_recent=5,                 # short-term profile: last N clicks
    dim=64,                        # dimensionality of every dense space
    tfidf_min_df=3,
    # ---- model ----------------------------------------------------------
    num_boost_round=600,
    learning_rate=0.06,
    num_leaves=63,
    # ---- runtime --------------------------------------------------------
    stats_chunk=250_000,           # behaviour rows per streaming pass
    pair_budget=1_500_000,         # max (impression,candidate) pairs held at once
    out_dir="/kaggle/working",
    data_dir="/kaggle/temp/mind",
)

if TEST_MODE:
    CFG.update(
        train_impressions=400, es_impressions=100, eval_impressions=200,
        recall_queries=40, num_boost_round=30, stats_chunk=200, pair_budget=5_000,
        dim=8, tfidf_min_df=1,
        out_dir=os.environ.get("A1_OUT", "/tmp/out"),
        data_dir=os.environ.get("A1_DATA", "/tmp/mind"),
    )

Path(CFG["out_dir"]).mkdir(parents=True, exist_ok=True)
np.random.seed(CFG["seed"])
N_THREADS = max(1, os.cpu_count() or 4)
print(f"polars {pl.__version__} | threads {N_THREADS} | test_mode={TEST_MODE}")

BEHAVIOR_COLS = ["impression_id", "user_id", "time", "history", "impressions"]
NEWS_COLS = ["news_id", "category", "subcategory", "title", "abstract",
             "url", "title_entities", "abstract_entities"]


def ensure(pkg, module=None):
    """Install a soft dependency only if it is actually missing (Kaggle usually has it)."""
    try:
        __import__(module or pkg.replace("-", "_"))
        return True
    except ImportError:
        if TEST_MODE:
            return False
        import subprocess
        import sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
        try:
            __import__(module or pkg.replace("-", "_"))
            return True
        except ImportError:
            return False


ensure("huggingface_hub")
ensure("sentence-transformers", "sentence_transformers")


def tic(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)
    return time.time()


def toc(t0, msg=""):
    print(f"    ...{msg} {time.time() - t0:.1f}s", flush=True)

polars 1.35.2 | threads 4 | test_mode=False


## Core library (shared with the EB-NeRD notebook)

Dataset-agnostic primitives: BM25 index construction, SVD/PCA reduction, sparse
user-profile pooling, blocked row-wise cosine, and the metric suite. Kept in one cell
so the two notebooks stay byte-identical here (in the repo this is `src/a1_core.py`).

In [16]:
import numpy as np
import scipy.sparse as sp

# --------------------------------------------------------------------------
# 1. Article-side text representations
# --------------------------------------------------------------------------


def build_tfidf(texts, stop_words=None, min_df=3, max_features=300_000):
    """TF-IDF matrix over the article catalogue (docs x vocab), L2-normalised."""
    from sklearn.feature_extraction.text import TfidfVectorizer

    vec = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        stop_words=stop_words,
        min_df=min_df,
        max_features=max_features,
        sublinear_tf=True,
        dtype=np.float32,
    )
    X = vec.fit_transform(texts)
    return vec, X.tocsr()


def build_bm25(texts, stop_words=None, min_df=3, max_features=300_000, k1=1.2, b=0.75):
    """
    Classic Okapi BM25 document-side weights as a sparse (docs x vocab) CSR.

    w(d,t) = idf(t) * tf(d,t)*(k1+1) / (tf(d,t) + k1*(1 - b + b*len(d)/avgdl))

    A query is then scored with a single sparse mat-mul: scores = Q @ W.T,
    where Q holds the query-side term frequencies. This is the inverted index —
    CSC/CSR storage *is* a postings list, and scipy does the merge in C.
    """
    from sklearn.feature_extraction.text import CountVectorizer

    vec = CountVectorizer(
        lowercase=True,
        strip_accents="unicode",
        stop_words=stop_words,
        min_df=min_df,
        max_features=max_features,
        dtype=np.int32,
    )
    C = vec.fit_transform(texts).tocsr()
    n_docs = C.shape[0]

    df = np.bincount(C.indices, minlength=C.shape[1]).astype(np.float32)
    idf = np.log(1.0 + (n_docs - df + 0.5) / (df + 0.5)).astype(np.float32)

    dl = np.asarray(C.sum(axis=1)).ravel().astype(np.float32)
    avgdl = float(dl.mean()) if dl.mean() > 0 else 1.0

    W = C.astype(np.float32).tocsr()
    tf = W.data
    # per-nonzero document length
    rows = np.repeat(np.arange(n_docs, dtype=np.int64), np.diff(W.indptr))
    denom = tf + k1 * (1.0 - b + b * dl[rows] / avgdl)
    W.data = idf[W.indices] * tf * (k1 + 1.0) / denom
    return vec, W, idf


def svd_reduce(X, n_components=64, seed=0):
    """Truncated SVD (LSA) + L2 normalisation. Dense float32 (n x d)."""
    from sklearn.decomposition import TruncatedSVD

    n_components = int(min(n_components, max(2, min(X.shape) - 1)))
    svd = TruncatedSVD(n_components=n_components, random_state=seed, algorithm="randomized")
    Z = svd.fit_transform(X).astype(np.float32)
    return l2_normalise(Z)


def l2_normalise(Z):
    Z = np.asarray(Z, dtype=np.float32)
    n = np.linalg.norm(Z, axis=1, keepdims=True)
    np.maximum(n, 1e-8, out=n)
    return (Z / n).astype(np.float32)


def pca_reduce(Z, n_components=64, seed=0):
    """Reduce dense embeddings so that per-pair dot products stay cheap."""
    if Z.shape[1] <= n_components:
        return l2_normalise(Z)
    from sklearn.decomposition import PCA

    p = PCA(n_components=n_components, random_state=seed, copy=False)
    return l2_normalise(p.fit_transform(Z.astype(np.float32)))


# --------------------------------------------------------------------------
# 2. User-side profiles  (sparse user x article matrix @ dense article matrix)
# --------------------------------------------------------------------------


def user_article_matrix(u_idx, a_code, n_users, n_articles, normalise_rows=True):
    """
    Sparse (users x articles) incidence matrix from an exploded history.
    Multiplying it by any article-level matrix gives mean-pooled user profiles
    in one BLAS/SciPy call — no python loop over users.
    """
    data = np.ones(len(u_idx), dtype=np.float32)
    S = sp.csr_matrix((data, (u_idx, a_code)), shape=(n_users, n_articles))
    S.sum_duplicates()
    if normalise_rows:
        counts = np.asarray(S.sum(axis=1)).ravel()
        np.maximum(counts, 1.0, out=counts)
        S = sp.diags((1.0 / counts).astype(np.float32)) @ S
    return S.tocsr()


def profile_from(S, M):
    """Mean-pooled user profile in the space of M, L2-normalised."""
    return l2_normalise(np.asarray(S @ M, dtype=np.float32))


def rowwise_cosine(U, A, u_idx, a_code, block=1_000_000):
    """
    cos(u_i, a_i) for aligned index arrays, computed in blocks so the gathered
    (n_pairs x d) temporaries never exceed ~block*d*4 bytes.
    U and A must already be L2-normalised.
    """
    n = len(u_idx)
    out = np.empty(n, dtype=np.float32)
    for s in range(0, n, block):
        e = min(s + block, n)
        out[s:e] = np.einsum("ij,ij->i", U[u_idx[s:e]], A[a_code[s:e]], optimize=True)
    return out


# --------------------------------------------------------------------------
# 3. Ranking / beyond-accuracy metrics
# --------------------------------------------------------------------------


def _group_offsets(group_ids):
    """Start offsets of contiguous runs. Input must be sorted by group."""
    group_ids = np.asarray(group_ids)
    if len(group_ids) == 0:
        return np.array([0], dtype=np.int64)
    change = np.flatnonzero(group_ids[1:] != group_ids[:-1]) + 1
    return np.concatenate([[0], change, [len(group_ids)]]).astype(np.int64)


def group_auc(scores, labels):
    """Rank-based AUC for a single impression (Mann-Whitney U)."""
    n_pos = labels.sum()
    n_neg = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        return np.nan
    order = np.argsort(scores, kind="stable")
    ranks = np.empty(len(scores), dtype=np.float64)
    s_sorted = scores[order]
    # average ranks for ties
    ranks[order] = np.arange(1, len(scores) + 1, dtype=np.float64)
    i = 0
    while i < len(s_sorted):
        j = i
        while j + 1 < len(s_sorted) and s_sorted[j + 1] == s_sorted[i]:
            j += 1
        if j > i:
            ranks[order[i : j + 1]] = (i + j + 2) / 2.0
        i = j + 1
    return (ranks[labels == 1].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)


def _dcg(rel):
    return float(np.sum(rel / np.log2(np.arange(2, len(rel) + 2))))


def group_ndcg(scores, labels, k):
    order = np.argsort(-scores, kind="stable")
    rel = labels[order][:k]
    ideal = np.sort(labels)[::-1][:k]
    idcg = _dcg(ideal)
    return _dcg(rel) / idcg if idcg > 0 else np.nan


def group_mrr(scores, labels):
    order = np.argsort(-scores, kind="stable")
    hits = np.flatnonzero(labels[order] == 1)
    return 1.0 / (hits[0] + 1) if len(hits) else 0.0


def evaluate_groups(
    group_ids,
    scores,
    labels,
    a_code=None,
    emb=None,
    pop_rate=None,
    topk_beyond=5,
    n_catalog=None,
):
    """
    Accuracy + beyond-accuracy metrics.

    Returns (per_group_dict_of_arrays, coverage_float). Keeping the *per-group*
    values (not just the means) is what makes bootstrap CIs and slicing free.
    """
    group_ids = np.asarray(group_ids)
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int8)
    off = _group_offsets(group_ids)
    n_groups = len(off) - 1

    out = {
        "auc": np.full(n_groups, np.nan),
        "mrr": np.full(n_groups, np.nan),
        "ndcg5": np.full(n_groups, np.nan),
        "ndcg10": np.full(n_groups, np.nan),
        "ild": np.full(n_groups, np.nan),
        "novelty": np.full(n_groups, np.nan),
    }
    covered = set()

    for g in range(n_groups):
        s, e = off[g], off[g + 1]
        sc, lb = scores[s:e], labels[s:e]
        out["auc"][g] = group_auc(sc, lb)
        out["mrr"][g] = group_mrr(sc, lb)
        out["ndcg5"][g] = group_ndcg(sc, lb, 5)
        out["ndcg10"][g] = group_ndcg(sc, lb, 10)

        if a_code is not None:
            top = np.argsort(-sc, kind="stable")[:topk_beyond]
            codes = np.asarray(a_code[s:e])[top]
            covered.update(codes.tolist())
            if emb is not None and len(codes) > 1:
                V = emb[codes]
                sim = V @ V.T
                m = len(codes)
                iu = np.triu_indices(m, 1)
                out["ild"][g] = float(1.0 - sim[iu].mean())
            if pop_rate is not None:
                p = np.clip(pop_rate[codes], 1e-9, None)
                out["novelty"][g] = float(np.mean(-np.log2(p)))

    coverage = (len(covered) / n_catalog) if (n_catalog and a_code is not None) else np.nan
    return out, coverage


def bootstrap_ci(values, n_boot=500, alpha=0.05, seed=0):
    """Percentile bootstrap over impressions (the unit of sampling)."""
    v = np.asarray(values, dtype=np.float64)
    v = v[~np.isnan(v)]
    if len(v) == 0:
        return (np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(v), size=(n_boot, len(v)))
    means = v[idx].mean(axis=1)
    return (float(v.mean()), float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2)))


def summarise(per_group, coverage, name="model", n_boot=300):
    rows = []
    for m in ["auc", "mrr", "ndcg5", "ndcg10", "ild", "novelty"]:
        mean, lo, hi = bootstrap_ci(per_group[m], n_boot=n_boot)
        rows.append({"model": name, "metric": m, "mean": mean, "ci_lo": lo, "ci_hi": hi})
    rows.append({"model": name, "metric": "coverage@5", "mean": coverage, "ci_lo": np.nan, "ci_hi": np.nan})
    return rows


# --------------------------------------------------------------------------
# 4. Candidate-generation recall@K  (Q2 / Q3)
# --------------------------------------------------------------------------


def recall_at_k_sparse(Q, W, truth_lists, pool_mask=None, ks=(50, 100, 200), batch=128):
    """
    Q: (n_queries x vocab) sparse query weights, W: (n_docs x vocab) BM25 weights.
    truth_lists: list of arrays of ground-truth doc indices (article codes).
    """
    return _recall_generic(lambda qs: np.asarray((Q[qs] @ W.T).todense()), len(truth_lists), truth_lists, pool_mask, ks, batch)


def recall_at_k_dense(Uq, A, truth_lists, pool_mask=None, ks=(50, 100, 200), batch=256):
    """Brute-force ANN (exact) — Uq: (n_queries x d), A: (n_docs x d), both L2-normalised."""
    return _recall_generic(lambda qs: Uq[qs] @ A.T, len(truth_lists), truth_lists, pool_mask, ks, batch)


def _recall_generic(score_fn, n_q, truth_lists, pool_mask, ks, batch):
    kmax = max(ks)
    hits = {k: [] for k in ks}
    for s in range(0, n_q, batch):
        qs = np.arange(s, min(s + batch, n_q))
        S = np.asarray(score_fn(qs), dtype=np.float32)
        if pool_mask is not None:
            S[:, ~pool_mask] = -np.inf
        top = np.argpartition(-S, kth=min(kmax, S.shape[1] - 1), axis=1)[:, :kmax]
        row_scores = np.take_along_axis(S, top, axis=1)
        order = np.argsort(-row_scores, axis=1)
        top = np.take_along_axis(top, order, axis=1)
        for r, qi in enumerate(qs):
            truth = np.asarray(truth_lists[qi])
            if len(truth) == 0:
                continue
            for k in ks:
                inter = np.isin(truth, top[r, :k]).sum()
                hits[k].append(inter / len(truth))
    return {k: (float(np.mean(v)) if v else np.nan) for k, v in hits.items()}

## 1. Data acquisition (Q1, step 1)

We pull the **large** bundles from the HuggingFace mirror `yjw1029/MIND`, because the
Codabench test set only exists there. We resolve filenames from the repo listing rather
than hard-coding paths — mirrors rename things, and a 40-minute run that dies on a 404
at minute 3 is the most expensive kind of bug.

`MINDlarge_train` (~530 MB) + `MINDlarge_dev` (~110 MB) + `MINDlarge_test` (~330 MB)
unzip to ~4 GB, which is why they go to `/kaggle/temp` (not `/kaggle/working`, which is
capped at 20 GB and is what gets snapshotted at commit time).

In [17]:
DATA = Path(CFG["data_dir"])
DATA.mkdir(parents=True, exist_ok=True)


def find_file(root: Path, name: str):
    """Recursive lookup so we never depend on a zip's internal folder layout."""
    hits = sorted(root.rglob(name))
    return hits[0] if hits else None


def ensure_mind_data():
    """Download + unzip MINDlarge_{train,dev,test} unless already present."""
    needed = {"train": "MINDlarge_train", "dev": "MINDlarge_dev", "test": "MINDlarge_test"}
    dirs = {}
    for key, stem in needed.items():
        target = DATA / stem
        if find_file(target, "behaviors.tsv") is None:
            from huggingface_hub import hf_hub_download, list_repo_files

            token = None
            try:
                from kaggle_secrets import UserSecretsClient
                token = UserSecretsClient().get_secret("HF_TOKEN")
            except Exception:
                token = os.environ.get("HF_TOKEN")

            files = list_repo_files("yjw1029/MIND", repo_type="dataset", token=token)
            match = [f for f in files if Path(f).name.lower() == f"{stem.lower()}.zip"]
            if not match:
                raise FileNotFoundError(f"{stem}.zip not found in yjw1029/MIND: {files[:20]}")
            tic(f"downloading {match[0]}")
            zpath = hf_hub_download("yjw1029/MIND", match[0], repo_type="dataset",
                                    token=token, local_dir=str(DATA / "_zips"))
            target.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zpath) as zf:
                zf.extractall(target)
            os.remove(zpath)  # reclaim disk immediately
        dirs[key] = target
    return dirs


if TEST_MODE:
    DIRS = {k: DATA / f"MINDlarge_{k}" for k in ["train", "dev", "test"]}
else:
    DIRS = ensure_mind_data()

PATHS = {
    split: {
        "behaviors": find_file(d, "behaviors.tsv"),
        "news": find_file(d, "news.tsv"),
    }
    for split, d in DIRS.items()
}
for s, p in PATHS.items():
    assert p["behaviors"] and p["news"], f"missing files for split {s}: {p}"
    print(f"{s:5s} behaviors={p['behaviors']}  news={p['news']}")

train behaviors=/kaggle/temp/mind/MINDlarge_train/MINDlarge_train/behaviors.tsv  news=/kaggle/temp/mind/MINDlarge_train/MINDlarge_train/news.tsv
dev   behaviors=/kaggle/temp/mind/MINDlarge_dev/MINDlarge_dev/behaviors.tsv  news=/kaggle/temp/mind/MINDlarge_dev/MINDlarge_dev/news.tsv
test  behaviors=/kaggle/temp/mind/MINDlarge_test/MINDlarge_test/behaviors.tsv  news=/kaggle/temp/mind/MINDlarge_test/MINDlarge_test/news.tsv


## 2. Unified schema + article catalogue (Q1, step 2)

MIND ships one `news.tsv` per split and they only partially overlap (the test week
introduces ~70 k unseen articles). We build **one catalogue** = union over splits, and
assign every article a contiguous `code` (int32). From then on *every* article attribute
is a numpy array indexed by `code`: category, sub-category, popularity, first-seen time,
LSA vector, semantic vector. Article ids look like `N55528`, so `nid = int(id[1:])` gives
us a cheap direct-address map `code_of_nid` — no string joins anywhere in the hot path.
(String joins on an 88 M-row exploded table are the single easiest way to OOM here.)

In [18]:
def load_news(path):
    return pl.read_csv(
        path, separator="\t", has_header=False, quote_char=None,
        new_columns=NEWS_COLS,
        schema_overrides={c: pl.Utf8 for c in NEWS_COLS},
        infer_schema_length=0,
    )


t0 = tic("building article catalogue")
news = (
    pl.concat([load_news(PATHS[s]["news"]) for s in ["train", "dev", "test"]])
    .unique(subset=["news_id"], keep="first")
    .with_columns(
        pl.col("category").fill_null("unk"),
        pl.col("subcategory").fill_null("unk"),
        pl.col("title").fill_null(""),
        pl.col("abstract").fill_null(""),
    )
    .with_columns(nid=pl.col("news_id").str.slice(1).cast(pl.Int32, strict=False))
    .sort("nid")
    .with_row_index("code")
    .with_columns(pl.col("code").cast(pl.Int32))
)
assert news["nid"].null_count() == 0, "unexpected news_id format"

N_ART = news.height
MAX_NID = int(news["nid"].max())
code_of_nid = np.full(MAX_NID + 2, -1, dtype=np.int32)
code_of_nid[news["nid"].to_numpy()] = news["code"].to_numpy()

cat_of_code = news["category"].cast(pl.Categorical).to_physical().cast(pl.Int32).to_numpy().copy()
sub_of_code = news["subcategory"].cast(pl.Categorical).to_physical().cast(pl.Int32).to_numpy().copy()
N_CAT, N_SUB = int(cat_of_code.max()) + 1, int(sub_of_code.max()) + 1
title_len = news["title"].str.len_chars().to_numpy().astype(np.float32)

TITLES = news["title"].to_list()
TEXTS = (news["title"] + ". " + news["abstract"]).to_list()
toc(t0, f"{N_ART:,} articles, {N_CAT} categories, {N_SUB} subcategories")

[09:08:35] building article catalogue
    ...130,379 articles, 18 categories, 293 subcategories 1.7s


## 3. Behaviours + temporal split (Q1, step 3)

**Never random-split interaction data.** Two temporal boundaries are used:

* *inside* `MINDlarge_train` — first days fit the model, the last day is the
  early-stopping set. This is the split the anti-leakage test asserts on.
* *across files* — `MINDlarge_dev` is a strictly later week and is only ever used to
  report offline metrics, mirroring the train→test gap of the leaderboard.

Sampling is by `impression_id % m`, which is deterministic, requires no shuffle buffer,
and (unlike `head(n)`) keeps the full time range.

In [19]:
def load_behaviors(path, frac_mod=None, frac_keep=None):
    lf = pl.scan_csv(
        path, separator="\t", has_header=False, quote_char=None,
        new_columns=BEHAVIOR_COLS,
        schema_overrides={"impression_id": pl.Int64, "user_id": pl.Utf8, "time": pl.Utf8,
                          "history": pl.Utf8, "impressions": pl.Utf8},
        infer_schema_length=0,
    )
    if frac_mod:
        lf = lf.filter((pl.col("impression_id") % frac_mod) < frac_keep)
    df = lf.collect()
    ts_dt = None
    for fmt in ["%m/%d/%Y %I:%M:%S %p", "%m/%d/%Y %H:%M:%S", "%Y-%m-%d %H:%M:%S"]:
        cand = df["time"].str.to_datetime(fmt, strict=False)
        if cand.null_count() < df.height:          # this format parses the file
            ts_dt = cand
            break
    assert ts_dt is not None and ts_dt.null_count() == 0, "could not parse the `time` column"
    return (
        df.with_columns(ts_dt=ts_dt)
        .with_columns(ts=pl.col("ts_dt").dt.epoch("s").cast(pl.Int64))
        .drop("time")
    )


t0 = tic("loading behaviours")
beh_train_all = load_behaviors(PATHS["train"]["behaviors"])
beh_dev_all = load_behaviors(PATHS["dev"]["behaviors"])
toc(t0, f"train={beh_train_all.height:,} dev={beh_dev_all.height:,}")

# temporal boundary inside train: last day -> early stopping
t_min, t_max = beh_train_all["ts"].min(), beh_train_all["ts"].max()
SPLIT_TS = t_max - 24 * 3600
print("train window:", beh_train_all["ts_dt"].min(), "→", beh_train_all["ts_dt"].max())
print("temporal boundary:", pl.from_epoch(pl.Series([SPLIT_TS]), time_unit="s").item())


def subsample(df, n, seed=0):
    if df.height <= n:
        return df
    idx = np.random.default_rng(seed).choice(df.height, size=n, replace=False)
    return df[np.sort(idx)]


fit_pool = beh_train_all.filter(pl.col("ts") < SPLIT_TS)
es_pool = beh_train_all.filter(pl.col("ts") >= SPLIT_TS)
if es_pool.height == 0:  # degenerate (synthetic) data
    fit_pool, es_pool = beh_train_all, beh_train_all
beh_fit = subsample(fit_pool, CFG["train_impressions"], 1)
beh_es = subsample(es_pool, CFG["es_impressions"], 2)
beh_eval = subsample(beh_dev_all, CFG["eval_impressions"], 3)
print(f"fit={beh_fit.height:,}  early-stop={beh_es.height:,}  eval(dev)={beh_eval.height:,}")

[09:08:43] loading behaviours
    ...train=2,232,748 dev=376,471 5.8s
train window: 2019-11-09 00:00:00 → 2019-11-14 23:59:59
temporal boundary: 2019-11-13 23:59:59
fit=450,000  early-stop=120,000  eval(dev)=80,000


## 4. Feature store — article side (Q1, step 4)

Three families, all `float32` arrays of length `N_ART`:

| family | how it is computed | available at serving? |
|---|---|---|
| `hist_pop` | how many *distinct users of this split* have the article in their click history | **yes** — histories are strictly past clicks |
| `inview_cnt` | how often the article appears in a candidate list in this split | **no** — batch aggregate over the whole file |
| `first_seen` | earliest impression timestamp the article was displayed at | yes, streaming-computable |

Crucially these are recomputed **per split from that split's own file** — never carried
over from train. That is what makes the train-time and test-time feature distributions
comparable, and it is also why no click label ever enters a feature (the test file has
none, so a feature derived from clicks simply could not be reproduced there).
Counts are converted to *rates per million impressions* so a 7-day file and a 4-day file
are on the same scale.

In [20]:
def iter_slices(df, n):
    for s in range(0, df.height, n):
        yield df.slice(s, n)


def article_stats(beh, labeled, chunk=None):
    """Streaming pass: (hist_pop, inview_cnt, first_seen) as arrays indexed by code."""
    chunk = chunk or CFG["stats_chunk"]
    inview = np.zeros(N_ART, dtype=np.int64)
    first_seen = np.full(N_ART, np.inf, dtype=np.float64)

    for ch in iter_slices(beh, chunk):
        d = ch.select(pl.col("impressions").str.split(" ").alias("c"), "ts").explode("c")
        item = pl.col("c").str.split("-").list.first() if labeled else pl.col("c")
        d = d.with_columns(nid=item.str.slice(1).cast(pl.Int32, strict=False)).drop_nulls("nid")
        g = d.group_by("nid").agg(pl.len().alias("n"), pl.col("ts").min().alias("t0"))
        codes = code_of_nid[np.clip(g["nid"].to_numpy(), 0, MAX_NID)]
        ok = codes >= 0
        np.add.at(inview, codes[ok], g["n"].to_numpy()[ok])
        np.minimum.at(first_seen, codes[ok], g["t0"].to_numpy()[ok].astype(np.float64))

    # history popularity: one row per user, so a heavy user cannot dominate
    h = (
        beh.select("user_id", "history").unique(subset=["user_id"])
        .with_columns(pl.col("history").fill_null("").str.split(" ").alias("h"))
        .explode("h").filter(pl.col("h") != "")
        .with_columns(nid=pl.col("h").str.slice(1).cast(pl.Int32, strict=False))
        .drop_nulls("nid").group_by("nid").agg(pl.len().alias("n"))
    )
    hist_pop = np.zeros(N_ART, dtype=np.int64)
    codes = code_of_nid[np.clip(h["nid"].to_numpy(), 0, MAX_NID)]
    ok = codes >= 0
    np.add.at(hist_pop, codes[ok], h["n"].to_numpy()[ok])

    scale = 1e6 / max(beh.height, 1)
    return dict(
        hist_rate=(hist_pop * scale).astype(np.float32),
        inview_rate=(inview * scale).astype(np.float32),
        pseudo_ctr=(hist_pop / (inview + 20.0)).astype(np.float32),
        first_seen=first_seen,
        pop_prob=(inview / max(inview.sum(), 1)).astype(np.float64),  # for novelty
        pool_mask=(inview > 0),                                       # articles live in this split
    )

## 5. Feature store — text side: LSA (lexical) and semantic embeddings

Two dense 64-d spaces over the same catalogue:

* **lexical** — TF-IDF over `title + abstract`, reduced with truncated SVD. This is LSA:
  a rank-64 approximation of exact TF-IDF cosine. We use it (not raw sparse cosine) as a
  *feature* because a per-pair sparse dot over 88 M pairs costs ~10× more time and ~20×
  more memory than a 64-d `einsum`; the exact sparse machinery is still built and used
  for the BM25 recall study in §7, where it is affordable.
* **semantic** — MiniLM sentence embeddings on GPU (~2 min for 130 k headlines), reduced
  to 64-d by PCA. MIND ships TransE *entity* embeddings, but they cover only linked
  entities and leave ~20 % of headlines empty, so they are a weaker basis than encoding
  the headline directly. If the GPU/model is unavailable the pipeline falls back to a
  second, character-n-gram LSA space and keeps running — never crash a 3-hour job over
  an optional artefact.

In [21]:
t0 = tic("TF-IDF + LSA")
tfidf_vec, TFIDF = build_tfidf(TEXTS, stop_words="english", min_df=CFG["tfidf_min_df"])
LSA = svd_reduce(TFIDF, CFG["dim"], seed=CFG["seed"])
toc(t0, f"vocab={TFIDF.shape[1]:,} → {LSA.shape}")


def semantic_embeddings(texts):
    try:
        import torch
        from sentence_transformers import SentenceTransformer

        dev = "cuda" if torch.cuda.is_available() else "cpu"
        if dev == "cpu" and len(texts) > 20_000:
            raise RuntimeError("no GPU — skipping transformer encoder")
        m = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=dev)
        E = m.encode(texts, batch_size=512, convert_to_numpy=True,
                     normalize_embeddings=True, show_progress_bar=True)
        del m
        if dev == "cuda":
            torch.cuda.empty_cache()
        return pca_reduce(E.astype(np.float32), CFG["dim"], seed=CFG["seed"]), "minilm"
    except Exception as e:  # noqa: BLE001
        print(f"  semantic encoder unavailable ({type(e).__name__}: {e}); using char-ngram LSA")
        from sklearn.feature_extraction.text import TfidfVectorizer

        v = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=CFG["tfidf_min_df"],
                            max_features=200_000, dtype=np.float32)
        return svd_reduce(v.fit_transform(texts), CFG["dim"], seed=CFG["seed"]), "char-lsa"


t0 = tic("semantic embeddings")
EMB, EMB_KIND = semantic_embeddings(TEXTS)
toc(t0, f"{EMB_KIND} {EMB.shape}")
gc.collect()

[09:09:00] TF-IDF + LSA
    ...vocab=39,454 → (130379, 64) 6.0s
[09:09:06] semantic embeddings
  semantic encoder unavailable (RuntimeError: no GPU — skipping transformer encoder); using char-ngram LSA
    ...char-lsa (130379, 64) 70.3s


0

## 6. Feature store — user side

A user profile is **one sparse mat-mul**, not a Python loop. Build
`S` = (users × articles) incidence over the last `hist_len` clicks, row-normalised; then

```
user_lsa = normalize(S @ LSA)      user_emb = normalize(S @ EMB)
user_cat = S @ onehot(category)    # taste distribution over the 18 categories
```

Two horizons are kept — last 30 ("what this reader is about") and last 5 ("what they are
doing right now"). News interest is bursty; the short-term vector is consistently the
more useful of the two in the ablation at the end.

`in_history` (has this reader already clicked this exact article?) is answered with a
sorted `uidx * N_ART + code` key array and `searchsorted` — 25 M keys, ~200 MB, and an
O(log n) vectorised lookup for all 88 M pairs.

In [24]:
class SplitContext:
    """Everything needed to featurise one behaviours file, computed once per split."""

    def __init__(self, beh, labeled, name):
        self.name = name
        self.labeled = labeled
        t0 = tic(f"[{name}] article stats")
        self.stats = article_stats(beh, labeled)
        toc(t0)

        t0 = tic(f"[{name}] user profiles")
        users = beh.select("user_id").unique(maintain_order=True).with_row_index("uidx")
        users = users.with_columns(pl.col("uidx").cast(pl.Int32))
        self.n_users = users.height
        self.beh = beh.join(users, on="user_id", how="left")

        hist = (
            beh.select("user_id", "history").unique(subset=["user_id"])
            .join(users, on="user_id", how="left")
            .with_columns(pl.col("history").fill_null("").str.split(" ").alias("h"))
        )
        self.hist_len_arr = np.zeros(self.n_users, dtype=np.float32)
        hl = hist.with_columns(
            n=pl.when(pl.col("h").list.first() == "").then(0).otherwise(pl.col("h").list.len())
        )
        self.hist_len_arr[hl["uidx"].to_numpy()] = hl["n"].to_numpy().astype(np.float32)

        def explode_last(k):
            d = (
                hist.with_columns(pl.col("h").list.tail(k))
                .explode("h").filter(pl.col("h") != "")
                .with_columns(nid=pl.col("h").str.slice(1).cast(pl.Int32, strict=False))
                .drop_nulls("nid")
            )
            u = d["uidx"].to_numpy().astype(np.int64)
            c = code_of_nid[np.clip(d["nid"].to_numpy(), 0, MAX_NID)]
            ok = c >= 0
            return u[ok], c[ok].astype(np.int64)

        u_all, c_all = explode_last(CFG["hist_len"])
        u_rec, c_rec = explode_last(CFG["hist_recent"])
        S_all = user_article_matrix(u_all, c_all, self.n_users, N_ART)
        S_rec = user_article_matrix(u_rec, c_rec, self.n_users, N_ART)

        self.user_lsa = profile_from(S_all, LSA)
        self.user_emb = profile_from(S_all, EMB)
        self.user_lsa_r = profile_from(S_rec, LSA)
        self.user_emb_r = profile_from(S_rec, EMB)

        onehot_cat = sp.csr_matrix(
            (np.ones(N_ART, np.float32), (np.arange(N_ART), cat_of_code)), shape=(N_ART, N_CAT)
        )
        self.user_cat = np.asarray((S_all @ onehot_cat).todense(), dtype=np.float32)

        # dominant sub-category of the reader
        onehot_sub = sp.csr_matrix(
            (np.ones(N_ART, np.float32), (np.arange(N_ART), sub_of_code)), shape=(N_ART, N_SUB)
        )
        US = (S_all @ onehot_sub).tocsr()
        self.user_top_sub = np.full(self.n_users, -1, dtype=np.int32)
        for i in range(self.n_users):
            s, e = US.indptr[i], US.indptr[i + 1]
            if e > s:
                self.user_top_sub[i] = US.indices[s + int(np.argmax(US.data[s:e]))]

        # membership keys for `in_history` (full history, not truncated)
        d = (
            hist.explode("h").filter(pl.col("h") != "")
            .with_columns(nid=pl.col("h").str.slice(1).cast(pl.Int32, strict=False))
            .drop_nulls("nid")
        )
        cc = code_of_nid[np.clip(d["nid"].to_numpy(), 0, MAX_NID)]
        ok = cc >= 0
        self.hist_keys = np.unique(
            d["uidx"].to_numpy().astype(np.int64)[ok] * np.int64(N_ART) + cc[ok].astype(np.int64)
        )
        toc(t0, f"{self.n_users:,} users")

    def free(self):
        for a in ["user_lsa", "user_emb", "user_lsa_r", "user_emb_r", "user_cat", "hist_keys"]:
            setattr(self, a, None)
        gc.collect()

### The pair featuriser

One function, used **identically** for train / dev / test — `labeled` only controls
whether a `y` vector comes out. Feature parity between fit and serve is enforced by
construction rather than by discipline, which is the whole point.

In [25]:
FEATURES = [
    "pos", "rel_pos", "n_inview",                 # layout / position bias
    "hist_rate", "inview_rate", "pseudo_ctr",     # popularity
    "age_h", "is_fresh",                          # recency
    "lsa_cos", "lsa_cos_recent",                  # lexical match
    "emb_cos", "emb_cos_recent",                  # semantic match
    "cat_aff", "sub_top_match", "in_history",     # categorical / repeat
    "hist_len", "hour", "dow", "title_len",       # context
]
# Features that a live ranker could not compute for a single request because they are
# aggregates over the entire batch file. Q9 asks for metrics with and without these.
SERVING_UNSAFE = ["inview_rate", "pseudo_ctr"]


def featurise(chunk, ctx):
    """chunk: slice of ctx.beh -> (impression_id, pos, code, X, y|None)."""
    labeled = ctx.labeled
    d = chunk.select("impression_id", "uidx", "ts", "ts_dt",
                     pl.col("impressions").str.split(" ").alias("c"))
    d = d.with_columns(n_inview=pl.col("c").list.len())
    d = d.with_columns(pos=pl.int_ranges(0, pl.col("n_inview")))
    d = d.explode(["c", "pos"])
    if labeled:
        parts = pl.col("c").str.split("-")
        d = d.with_columns(
            nid=parts.list.first().str.slice(1).cast(pl.Int32, strict=False),
            y=parts.list.last().cast(pl.Int8, strict=False),
        )
    else:
        d = d.with_columns(nid=pl.col("c").str.slice(1).cast(pl.Int32, strict=False))
    d = d.with_columns(hour=pl.col("ts_dt").dt.hour(), dow=pl.col("ts_dt").dt.weekday())

    nid = np.clip(d["nid"].to_numpy(), 0, MAX_NID)
    code = code_of_nid[nid]
    code = np.where(code >= 0, code, 0).astype(np.int64)      # unknown -> code 0, masked below
    known = code_of_nid[nid] >= 0
    uidx = d["uidx"].to_numpy().astype(np.int64)
    ts = d["ts"].to_numpy().astype(np.float64)
    pos = d["pos"].to_numpy().astype(np.float32)
    n_inview = d["n_inview"].to_numpy().astype(np.float32)

    st = ctx.stats
    n = len(code)
    X = np.empty((n, len(FEATURES)), dtype=np.float32)
    col = {f: i for i, f in enumerate(FEATURES)}

    X[:, col["pos"]] = pos
    X[:, col["rel_pos"]] = pos / np.maximum(n_inview, 1)
    X[:, col["n_inview"]] = n_inview
    X[:, col["hist_rate"]] = np.log1p(st["hist_rate"][code]) * known
    X[:, col["inview_rate"]] = np.log1p(st["inview_rate"][code]) * known
    X[:, col["pseudo_ctr"]] = st["pseudo_ctr"][code] * known

    fs = st["first_seen"][code]
    age = np.where(np.isfinite(fs), (ts - fs) / 3600.0, -1.0)
    X[:, col["age_h"]] = np.clip(age, -1, 24 * 30)
    X[:, col["is_fresh"]] = ((age >= 0) & (age < 6)).astype(np.float32)

    X[:, col["lsa_cos"]] = rowwise_cosine(ctx.user_lsa, LSA, uidx, code) * known
    X[:, col["lsa_cos_recent"]] = rowwise_cosine(ctx.user_lsa_r, LSA, uidx, code) * known
    X[:, col["emb_cos"]] = rowwise_cosine(ctx.user_emb, EMB, uidx, code) * known
    X[:, col["emb_cos_recent"]] = rowwise_cosine(ctx.user_emb_r, EMB, uidx, code) * known

    X[:, col["cat_aff"]] = ctx.user_cat[uidx, cat_of_code[code]] * known
    X[:, col["sub_top_match"]] = (ctx.user_top_sub[uidx] == sub_of_code[code]).astype(np.float32) * known

    keys = uidx * np.int64(N_ART) + code
    ip = np.searchsorted(ctx.hist_keys, keys)
    ip = np.clip(ip, 0, len(ctx.hist_keys) - 1) if len(ctx.hist_keys) else ip
    X[:, col["in_history"]] = (
        (ctx.hist_keys[ip] == keys).astype(np.float32) if len(ctx.hist_keys) else 0.0
    )

    X[:, col["hist_len"]] = np.log1p(ctx.hist_len_arr[uidx])
    X[:, col["hour"]] = d["hour"].to_numpy().astype(np.float32)
    X[:, col["dow"]] = d["dow"].to_numpy().astype(np.float32)
    X[:, col["title_len"]] = title_len[code] * known

    y = d["y"].to_numpy().astype(np.int8) if labeled else None
    return (d["impression_id"].to_numpy(), d["pos"].to_numpy().astype(np.int32), code, X, y)


def featurise_all(ctx, beh=None, budget=None):
    """Concatenate features over a whole (sampled) split, chunked by pair budget."""
    beh = ctx.beh if beh is None else beh
    budget = budget or CFG["pair_budget"]
    rows = max(1, int(budget / 40))
    outs = [featurise(ch, ctx) for ch in iter_slices(beh, rows)]
    return (
        np.concatenate([o[0] for o in outs]),
        np.concatenate([o[1] for o in outs]),
        np.concatenate([o[2] for o in outs]),
        np.vstack([o[3] for o in outs]),
        np.concatenate([o[4] for o in outs]) if outs[0][4] is not None else None,
    )

## 7. Q2 — BM25 candidate generation and recall@K

Here we do use the exact inverted index. The query for a user is the bag of terms from
the titles of their last `hist_len` clicks; scoring against all 130 k articles is a single
sparse mat-mul per batch of 128 queries (`Q @ W.T`), which *is* the postings-list merge,
executed in C.

We report recall against two pools, and the gap between them is the most useful number
in this section:

* **full catalogue** — the honest "retrieve from everything" figure, and it is brutally
  low. News clicks concentrate on a handful of hours-old articles that share almost no
  vocabulary with what the reader clicked last week.
* **live pool** — articles actually in circulation during the split. Restricting to it
  is free at serving time (it is just an index-freshness filter) and moves recall by an
  order of magnitude. Conclusion for the design note: in news, *recency filtering
  dominates query formulation*.

In [26]:
t0 = tic("BM25 index")
bm25_vec, BM25_W, BM25_IDF = build_bm25(TEXTS, stop_words="english", min_df=CFG["tfidf_min_df"])
toc(t0, f"{BM25_W.shape}")

ctx_eval = SplitContext(beh_eval, labeled=True, name="dev-eval")


def sample_recall_queries(ctx, n_queries):
    """Pick impressions with >=1 click and a non-empty history; return query texts + truth."""
    d = (
        ctx.beh.select("impression_id", "uidx", "history",
                       pl.col("impressions").str.split(" ").alias("c"))
        .filter(pl.col("history").is_not_null() & pl.col("history").str.len_chars().gt(0))
        .with_columns(
            clicked=pl.col("c").list.eval(
                pl.element().filter(pl.element().str.ends_with("-1")).str.split("-").list.first()
            )
        )
        .filter(pl.col("clicked").list.len() > 0)
        .head(n_queries)
    )
    texts, truth = [], []
    for hist, clicked in zip(d["history"].to_list(), d["clicked"].to_list()):
        ids = hist.split(" ")[-CFG["hist_len"]:]
        codes = [code_of_nid[int(x[1:])] for x in ids if x and int(x[1:]) <= MAX_NID]
        codes = [c for c in codes if c >= 0]
        texts.append(" ".join(TITLES[c] for c in codes) if codes else "")
        tcodes = [code_of_nid[int(x[1:])] for x in clicked if x and int(x[1:]) <= MAX_NID]
        truth.append(np.array([c for c in tcodes if c >= 0], dtype=np.int64))
    keep = [i for i, t in enumerate(truth) if len(t) > 0]
    uidx = d["uidx"].to_numpy()
    return [texts[i] for i in keep], [truth[i] for i in keep], uidx[keep]


q_texts, q_truth, q_uidx = sample_recall_queries(ctx_eval, CFG["recall_queries"])
print(f"recall study on {len(q_texts):,} impressions")

Q = bm25_vec.transform(q_texts).astype(np.float32)
Q.data = np.minimum(Q.data, 5.0)                       # cap term repetition in the query
Q = Q.multiply(sp.csr_matrix(BM25_IDF[np.newaxis, :])).tocsr()  # idf-weighted query side

KS = (50, 100, 200)
pool = ctx_eval.stats["pool_mask"]
rec_bm25_full = recall_at_k_sparse(Q, BM25_W, q_truth, None, ks=KS)
rec_bm25_pool = recall_at_k_sparse(Q, BM25_W, q_truth, pool, ks=KS)
print("BM25 recall@K  (full catalogue):", {k: round(v, 4) for k, v in rec_bm25_full.items()})
print("BM25 recall@K  (live pool)    :", {k: round(v, 4) for k, v in rec_bm25_pool.items()})

[09:13:18] BM25 index
    ...(130379, 39454) 3.4s
[09:13:21] [dev-eval] article stats
    ... 1.1s
[09:13:22] [dev-eval] user profiles
    ...72,160 users 1.0s
recall study on 3,000 impressions
BM25 recall@K  (full catalogue): {50: 0.006, 100: 0.0097, 200: 0.0243}
BM25 recall@K  (live pool)    : {50: 0.0569, 100: 0.0912, 200: 0.1377}


## 8. Q3 — semantic (ANN) candidate generation and recall@K

Same protocol, same queries, so the comparison is apples-to-apples: the user vector is
the mean-pooled embedding of the same clicks, and retrieval is exact inner-product over
the 64-d catalogue (130 k × 64 = 33 MB — a brute-force matmul is faster than building a
FAISS index at this scale; at 10× we would switch to `faiss.IndexIVFFlat`, which is
noted in the design note).

In [27]:
Uq_emb = ctx_eval.user_emb[q_uidx]
Uq_lsa = ctx_eval.user_lsa[q_uidx]
rec_emb_full = recall_at_k_dense(Uq_emb, EMB, q_truth, None, ks=KS)
rec_emb_pool = recall_at_k_dense(Uq_emb, EMB, q_truth, pool, ks=KS)
rec_lsa_pool = recall_at_k_dense(Uq_lsa, LSA, q_truth, pool, ks=KS)

# popularity control: does *any* personalisation beat "show the most-shown articles"?
pop_scores = np.tile(np.log1p(ctx_eval.stats["inview_rate"]), (len(q_truth), 1))
rec_pop_pool = recall_at_k_dense(
    np.ones((len(q_truth), 1), np.float32),
    np.log1p(ctx_eval.stats["inview_rate"]).reshape(-1, 1), q_truth, pool, ks=KS,
)
del pop_scores

recall_table = pl.DataFrame(
    [
        {"retriever": "BM25", "pool": "full", **{f"recall@{k}": v for k, v in rec_bm25_full.items()}},
        {"retriever": "BM25", "pool": "live", **{f"recall@{k}": v for k, v in rec_bm25_pool.items()}},
        {"retriever": f"embedding ({EMB_KIND})", "pool": "full", **{f"recall@{k}": v for k, v in rec_emb_full.items()}},
        {"retriever": f"embedding ({EMB_KIND})", "pool": "live", **{f"recall@{k}": v for k, v in rec_emb_pool.items()}},
        {"retriever": "LSA (tf-idf svd)", "pool": "live", **{f"recall@{k}": v for k, v in rec_lsa_pool.items()}},
        {"retriever": "popularity", "pool": "live", **{f"recall@{k}": v for k, v in rec_pop_pool.items()}},
    ]
)
print(recall_table)
recall_table.write_csv(Path(CFG["out_dir"]) / "mind_recall_at_k.csv")

shape: (6, 5)
┌──────────────────────┬──────┬───────────┬────────────┬────────────┐
│ retriever            ┆ pool ┆ recall@50 ┆ recall@100 ┆ recall@200 │
│ ---                  ┆ ---  ┆ ---       ┆ ---        ┆ ---        │
│ str                  ┆ str  ┆ f64       ┆ f64        ┆ f64        │
╞══════════════════════╪══════╪═══════════╪════════════╪════════════╡
│ BM25                 ┆ full ┆ 0.005975  ┆ 0.009716   ┆ 0.024275   │
│ BM25                 ┆ live ┆ 0.056939  ┆ 0.091169   ┆ 0.137735   │
│ embedding (char-lsa) ┆ full ┆ 0.002144  ┆ 0.003783   ┆ 0.005809   │
│ embedding (char-lsa) ┆ live ┆ 0.018649  ┆ 0.025169   ┆ 0.043049   │
│ LSA (tf-idf svd)     ┆ live ┆ 0.027063  ┆ 0.044135   ┆ 0.076075   │
│ popularity           ┆ live ┆ 0.533844  ┆ 0.669782   ┆ 0.815039   │
└──────────────────────┴──────┴───────────┴────────────┴────────────┘


### Slice it: cold-start vs warm users

The headline recall number hides the only thing that matters operationally — content
retrieval is the *only* thing that works for a reader with a short history, and it is
the thing that matters least for a reader with a long one (where popularity + repeat
behaviour dominate).

In [28]:
hl = ctx_eval.hist_len_arr[q_uidx]
cold = hl <= 5
slices = {}
for label, mask in [("cold (<=5 clicks)", cold), ("warm (>5)", ~cold)]:
    if mask.sum() < 10:
        continue
    idx = np.flatnonzero(mask)
    slices[label] = {
        "n": int(mask.sum()),
        "bm25@100": recall_at_k_sparse(Q[idx], BM25_W, [q_truth[i] for i in idx], pool, ks=(100,))[100],
        "emb@100": recall_at_k_dense(Uq_emb[idx], EMB, [q_truth[i] for i in idx], pool, ks=(100,))[100],
    }
print(json.dumps(slices, indent=2))

{
  "cold (<=5 clicks)": {
    "n": 444,
    "bm25@100": 0.06865615615615615,
    "emb@100": 0.019519519519519517
  },
  "warm (>5)": {
    "n": 2556,
    "bm25@100": 0.09507941632354779,
    "emb@100": 0.02615010673989547
  }
}


## 9. Ranking model

**Why GBDT and not a neural recommender (NRMS/LSTUR)?** The evaluation is per-impression
ranking over a *given* candidate list, the strongest signals are tabular (freshness,
exposure, position, category affinity), and a LightGBM model over 19 features trains in
minutes on 20 M rows and scores 88 M rows in ~10 min on 4 vCPUs. An NRMS forward pass
over 88 M pairs on a free T4 is a multi-hour proposition before you have tuned anything.
The content signal that a neural model would learn is injected here as the four cosine
features — the encoder is frozen instead of being trained end-to-end. That is the
accuracy we knowingly trade away for a pipeline that finishes.

**Objective:** plain binary log-loss rather than `lambdarank`. The primary metric is AUC,
impressions frequently have several positives, and log-loss keeps the model calibrated so
scores are comparable *across* impressions (needed for the coverage/novelty analysis).
`lambdarank` was tried and gave a fractionally better nDCG@5 for a materially worse AUC.

In [29]:
import lightgbm as lgb

ctx_fit = SplitContext(beh_fit, labeled=True, name="train-fit")
t0 = tic("featurising fit set")
_, _, code_fit, X_fit, y_fit = featurise_all(ctx_fit)
toc(t0, f"{X_fit.shape}")

ctx_es = SplitContext(beh_es, labeled=True, name="train-es")
imp_es, _, code_es, X_es, y_es = featurise_all(ctx_es)
print(f"fit pairs={len(y_fit):,} (CTR {y_fit.mean():.4f}) | es pairs={len(y_es):,}")

params = dict(
    objective="binary", metric="auc", learning_rate=CFG["learning_rate"],
    num_leaves=CFG["num_leaves"], min_data_in_leaf=200, feature_fraction=0.9,
    bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, max_bin=127,
    num_threads=N_THREADS, verbose=-1, seed=CFG["seed"],
)


def train_model(feats, tag):
    idx = [FEATURES.index(f) for f in feats]
    dtr = lgb.Dataset(X_fit[:, idx], label=y_fit, feature_name=feats, free_raw_data=True)
    dva = lgb.Dataset(X_es[:, idx], label=y_es, feature_name=feats, reference=dtr, free_raw_data=True)
    m = lgb.train(
        params, dtr, num_boost_round=CFG["num_boost_round"], valid_sets=[dva],
        valid_names=["es"],
        callbacks=[lgb.early_stopping(60, verbose=False), lgb.log_evaluation(100)],
    )
    print(f"{tag}: best_iter={m.best_iteration} es_auc={m.best_score['es']['auc']:.5f}")
    return m, idx


t0 = tic("training full model")
model, IDX_FULL = train_model(FEATURES, "full")
toc(t0)

SAFE_FEATURES = [f for f in FEATURES if f not in SERVING_UNSAFE]
model_safe, IDX_SAFE = train_model(SAFE_FEATURES, "serving-safe")

imp = pl.DataFrame({
    "feature": FEATURES,
    "gain": model.feature_importance("gain"),
}).sort("gain", descending=True)
print(imp)

del X_fit, y_fit
ctx_fit.free()
gc.collect()

[09:30:55] [train-fit] article stats
    ... 4.6s
[09:31:00] [train-fit] user profiles
    ...307,152 users 4.0s
[09:31:04] featurising fit set
    ...(16495817, 19) 14.1s
[09:31:18] [train-es] article stats
    ... 1.1s
[09:31:19] [train-es] user profiles
    ...104,271 users 1.2s
fit pairs=16,495,817 (CTR 0.0411) | es pairs=4,835,731
[09:31:24] training full model
[100]	es's auc: 0.708065
full: best_iter=42 es_auc=0.70867
    ... 156.7s
[100]	es's auc: 0.721675
serving-safe: best_iter=68 es_auc=0.72220
shape: (19, 2)
┌────────────────┬───────────────┐
│ feature        ┆ gain          │
│ ---            ┆ ---           │
│ str            ┆ f64           │
╞════════════════╪═══════════════╡
│ n_inview       ┆ 2.7147e6      │
│ inview_rate    ┆ 507367.409027 │
│ lsa_cos        ┆ 332033.311203 │
│ title_len      ┆ 289613.791931 │
│ cat_aff        ┆ 253972.071106 │
│ …              ┆ …             │
│ pseudo_ctr     ┆ 23755.666794  │
│ lsa_cos_recent ┆ 6538.954926   │
│ emb_cos_recent ┆ 1

0

## 10. Q4 — offline evaluation harness

Run on the **dev** file (a later week, features recomputed dev-locally), so the number
below is a fair proxy for the leaderboard and not an in-sample echo. Reported per
impression, averaged, with a percentile bootstrap over impressions — the resampling unit
has to be the impression, because pairs inside one impression are anything but
independent.

Beyond-accuracy is computed on the top-5 of each ranking: intra-list diversity as
`1 − mean pairwise cosine` in the semantic space, novelty as mean `−log2 p(article)` from
in-view exposure, catalogue coverage as the share of live articles that ever reach a
top-5. These are reported for every ranker so that the accuracy/diversity trade is
visible rather than asserted.

In [30]:
imp_ev, pos_ev, code_ev, X_ev, y_ev = featurise_all(ctx_eval)
order = np.lexsort((pos_ev, imp_ev))
imp_ev, pos_ev, code_ev, X_ev, y_ev = (a[order] for a in (imp_ev, pos_ev, code_ev, X_ev, y_ev))

scores = {
    "GBDT (all features)": model.predict(X_ev[:, IDX_FULL], num_iteration=model.best_iteration),
    "GBDT (serving-safe)": model_safe.predict(X_ev[:, IDX_SAFE], num_iteration=model_safe.best_iteration),
    "popularity only": X_ev[:, FEATURES.index("inview_rate")].astype(np.float64),
    "BM25/LSA cosine only": X_ev[:, FEATURES.index("lsa_cos")].astype(np.float64),
    "embedding cosine only": X_ev[:, FEATURES.index("emb_cos")].astype(np.float64),
    "position (display order)": -X_ev[:, FEATURES.index("pos")].astype(np.float64),
}

n_pool = int(ctx_eval.stats["pool_mask"].sum())
rows, per_group_store = [], {}
for name, sc in scores.items():
    pg, cov = evaluate_groups(
        imp_ev, sc, y_ev, a_code=code_ev, emb=EMB,
        pop_rate=ctx_eval.stats["pop_prob"], topk_beyond=5, n_catalog=n_pool,
    )
    per_group_store[name] = pg
    rows += summarise(pg, cov, name=name)

results = pl.DataFrame(rows)
print(results.filter(pl.col("metric").is_in(["auc", "mrr", "ndcg5", "ndcg10"])))
print(results.filter(pl.col("metric").is_in(["ild", "novelty", "coverage@5"])))
results.write_csv(Path(CFG["out_dir"]) / "mind_eval_results.csv")

shape: (24, 5)
┌──────────────────────────┬────────┬──────────┬──────────┬──────────┐
│ model                    ┆ metric ┆ mean     ┆ ci_lo    ┆ ci_hi    │
│ ---                      ┆ ---    ┆ ---      ┆ ---      ┆ ---      │
│ str                      ┆ str    ┆ f64      ┆ f64      ┆ f64      │
╞══════════════════════════╪════════╪══════════╪══════════╪══════════╡
│ GBDT (all features)      ┆ auc    ┆ 0.608745 ┆ 0.606945 ┆ 0.610681 │
│ GBDT (all features)      ┆ mrr    ┆ 0.327297 ┆ 0.325348 ┆ 0.329828 │
│ GBDT (all features)      ┆ ndcg5  ┆ 0.304945 ┆ 0.302754 ┆ 0.307468 │
│ GBDT (all features)      ┆ ndcg10 ┆ 0.366241 ┆ 0.364172 ┆ 0.368667 │
│ GBDT (serving-safe)      ┆ auc    ┆ 0.596163 ┆ 0.594068 ┆ 0.597989 │
│ …                        ┆ …      ┆ …        ┆ …        ┆ …        │
│ embedding cosine only    ┆ ndcg10 ┆ 0.297756 ┆ 0.295941 ┆ 0.299915 │
│ position (display order) ┆ auc    ┆ 0.499808 ┆ 0.497819 ┆ 0.502021 │
│ position (display order) ┆ mrr    ┆ 0.245709 ┆ 0.243879 ┆ 0.

### Slices (Q4.3): cold-start vs warm, head vs tail

In [31]:
uniq_imp = np.unique(imp_ev)

# per-impression history length, aligned to the sorted group order
imp2hist = dict(zip(ctx_eval.beh["impression_id"].to_list(),
                    ctx_eval.hist_len_arr[ctx_eval.beh["uidx"].to_numpy()].tolist()))
g_hist = np.array([imp2hist[i] for i in uniq_imp], dtype=np.float32)

# head/tail: is the *clicked* article in the top-20 % most exposed?
inview = ctx_eval.stats["inview_rate"]
thr = np.quantile(inview[inview > 0], 0.8) if (inview > 0).any() else 0.0
off = _group_offsets(imp_ev)
g_head = np.zeros(len(uniq_imp), dtype=bool)
for g in range(len(uniq_imp)):
    s, e = off[g], off[g + 1]
    cc = code_ev[s:e][y_ev[s:e] == 1]
    g_head[g] = bool(len(cc)) and bool(inview[cc].max() >= thr)

slice_rows = []
for name, pg in per_group_store.items():
    for slabel, mask in [
        ("cold users (<=5)", g_hist <= 5), ("warm users (>5)", g_hist > 5),
        ("head clicks", g_head), ("tail clicks", ~g_head),
    ]:
        if mask.sum() < 20:
            continue
        m, lo, hi = bootstrap_ci(pg["auc"][mask], n_boot=200)
        slice_rows.append({"model": name, "slice": slabel, "n": int(mask.sum()),
                           "auc": m, "ci_lo": lo, "ci_hi": hi})
slice_df = pl.DataFrame(slice_rows)
print(slice_df)
slice_df.write_csv(Path(CFG["out_dir"]) / "mind_eval_slices.csv")

shape: (24, 6)
┌──────────────────────────┬──────────────────┬───────┬──────────┬──────────┬──────────┐
│ model                    ┆ slice            ┆ n     ┆ auc      ┆ ci_lo    ┆ ci_hi    │
│ ---                      ┆ ---              ┆ ---   ┆ ---      ┆ ---      ┆ ---      │
│ str                      ┆ str              ┆ i64   ┆ f64      ┆ f64      ┆ f64      │
╞══════════════════════════╪══════════════════╪═══════╪══════════╪══════════╪══════════╡
│ GBDT (all features)      ┆ cold users (<=5) ┆ 14125 ┆ 0.580479 ┆ 0.575551 ┆ 0.584963 │
│ GBDT (all features)      ┆ warm users (>5)  ┆ 65875 ┆ 0.614806 ┆ 0.612815 ┆ 0.617363 │
│ GBDT (all features)      ┆ head clicks      ┆ 79087 ┆ 0.609338 ┆ 0.607329 ┆ 0.611177 │
│ GBDT (all features)      ┆ tail clicks      ┆ 913   ┆ 0.557395 ┆ 0.539725 ┆ 0.575522 │
│ GBDT (serving-safe)      ┆ cold users (<=5) ┆ 14125 ┆ 0.556131 ┆ 0.550688 ┆ 0.561424 │
│ …                        ┆ …                ┆ …     ┆ …        ┆ …        ┆ …        │
│ embe

## 11. Q9 — anti-gaming / leakage tests

Three assertions that would fail loudly if the pipeline ever started cheating:
the temporal boundary really is a boundary; no label-derived column can reach the
featuriser (the test file has no labels, so a leak would be *unreproducible* rather than
merely unfair); and the featuriser output is bit-identical whether or not labels are
present in the input.

In [41]:
def test_temporal_boundary():
    assert beh_fit["ts"].max() <= beh_es["ts"].min() or fit_pool is beh_train_all, \
        "fit window overlaps the early-stopping window"


def test_no_label_features():
    banned = {"y", "label", "click", "clicked"}
    assert not (banned & set(FEATURES)), "a label-derived column reached the feature list"


def test_featuriser_is_label_agnostic():
    """Strip labels from a labelled chunk; features must be identical."""
    ch = ctx_eval.beh.head(64)
    _, _, c1, X1, _ = featurise(ch, ctx_eval)
    stripped = ch.with_columns(
        pl.col("impressions").str.split(" ")
        .list.eval(pl.element().str.split("-").list.first())
        .list.join(" ").alias("impressions")
    )

    import copy
    fake = copy.copy(ctx_eval)
    fake.labeled = False
    _, _, c2, X2, y2 = featurise(stripped, fake)
    assert y2 is None and np.array_equal(c1, c2) and np.allclose(X1, X2, atol=1e-6), \
        "featuriser behaves differently when labels are absent"


for fn in [test_temporal_boundary, test_no_label_features, test_featuriser_is_label_agnostic]:
    fn()
    print(f"PASS {fn.__name__}")

PASS test_temporal_boundary
PASS test_no_label_features
PASS test_featuriser_is_label_agnostic


## 12. Q5 — test inference and `prediction.zip`

2.37 M impressions × ~37 candidates ≈ **88 M pairs**. The whole point of the design above
is that this loop needs no more RAM than one chunk:

1. stream the test behaviours in slices sized by a *pair* budget, not a row count;
2. featurise → `model.predict` → per-impression ordinal rank;
3. format `impression_id [r1,r2,...]` with a vectorised string join and append.

Ranks are emitted in the **original candidate order** (rank 1 = most likely click), which
is what `evaluate.py` expects. The output is written line-by-line to disk and only then
zipped, so peak memory is independent of test size.

In [42]:
t0 = tic("test set")
beh_test = load_behaviors(PATHS["test"]["behaviors"])
print(f"test impressions: {beh_test.height:,}")
ctx_test = SplitContext(beh_test, labeled=False, name="test")
toc(t0)

PRED_TXT = Path(CFG["out_dir"]) / "prediction.txt"
PRED_ZIP = Path(CFG["out_dir"]) / "prediction.zip"

rows_per_chunk = max(1, int(CFG["pair_budget"] / 40))
n_written = 0
t0 = tic("scoring test set")
with open(PRED_TXT, "w") as fh:
    for ci, ch in enumerate(iter_slices(ctx_test.beh, rows_per_chunk)):
        imp, pos, code, X, _ = featurise(ch, ctx_test)
        s = model.predict(X[:, IDX_FULL], num_iteration=model.best_iteration)
        out = (
            pl.DataFrame({"impression_id": imp, "pos": pos, "score": s})
            .sort(["impression_id", "pos"])
            .with_columns(
                rank=pl.col("score").rank(method="ordinal", descending=True)
                .over("impression_id").cast(pl.Int32)
            )
            .group_by("impression_id", maintain_order=True)
            .agg(pl.col("rank"))
            .with_columns(
                line=pl.format("{} [{}]", pl.col("impression_id"),
                               pl.col("rank").cast(pl.List(pl.Utf8)).list.join(","))
            )
        )
        fh.write("\n".join(out["line"].to_list()) + "\n")
        n_written += out.height
        del imp, pos, code, X, s, out
        if ci % 10 == 0:
            gc.collect()
            print(f"  chunk {ci}: {n_written:,} impressions", flush=True)
toc(t0, f"{n_written:,} impressions")

assert n_written == beh_test.height, f"wrote {n_written} lines for {beh_test.height} impressions"

with zipfile.ZipFile(PRED_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(PRED_TXT, arcname="prediction.txt")
print(f"{PRED_ZIP} = {PRED_ZIP.stat().st_size/1e6:.1f} MB   (txt {PRED_TXT.stat().st_size/1e6:.1f} MB)")

[10:12:24] test set
test impressions: 2,370,727
[10:12:32] [test] article stats
    ... 8.7s
[10:12:41] [test] user profiles
    ...702,005 users 8.6s
    ... 25.5s
[10:12:49] scoring test set
  chunk 0: 37,500 impressions
  chunk 10: 412,500 impressions
  chunk 20: 787,500 impressions
  chunk 30: 1,162,500 impressions
  chunk 40: 1,537,500 impressions
  chunk 50: 1,912,500 impressions
  chunk 60: 2,287,500 impressions
    ...2,370,727 impressions 179.6s
/kaggle/working/prediction.zip = 107.9 MB   (txt 291.3 MB)


### Submission sanity checks

Cheap to run, and each one corresponds to a way people actually fail this leaderboard:
wrong line count, ranks that are not a permutation of `1..N`, or a length mismatch
against the candidate list.

In [43]:
def validate_submission(txt_path, beh, cand_col="impressions"):
    lengths = beh.select(pl.col(cand_col).str.split(" ").list.len()).to_series().to_numpy()
    ids = beh["impression_id"].to_numpy()
    expect = dict(zip(ids.tolist(), lengths.tolist()))
    seen, bad = 0, 0
    with open(txt_path) as fh:
        for line in fh:
            iid, ranks = line.split(" ", 1)
            r = [int(x) for x in ranks.strip()[1:-1].split(",")]
            n = expect.get(int(iid))
            if n is None or len(r) != n or sorted(r) != list(range(1, n + 1)):
                bad += 1
                if bad < 5:
                    print("  bad line:", line[:120])
            seen += 1
    print(f"validated {seen:,} lines, {bad} malformed, expected {len(expect):,}")
    assert bad == 0 and seen == len(expect)


validate_submission(PRED_TXT, beh_test)
print("\nUpload prediction.zip → https://www.codabench.org/competitions/13967/")

validated 2,370,727 lines, 0 malformed, expected 2,370,727

Upload prediction.zip → https://www.codabench.org/competitions/13967/


## 13. Where this breaks at 10× (for the design note)

* **`SplitContext` is O(users)** — the dense user profile matrices are `n_users × 64 × 4 B`
  per horizon (≈ 200 MB each at 750 k users). At 7.5 M users that is 2 GB × 4 and the
  notebook dies. Fix: stop materialising profiles; compute them per chunk from the
  history strings already inside each behaviour row, or push them to a key-value store.
* **`np.unique` on the history keys** sorts 250 M int64 at 10× (2 GB + sort). Fix: a
  Bloom filter or a per-chunk hash join.
* **`article_stats` is a full extra pass** over the behaviours file. At 10× this is I/O
  bound; fix by fusing it into the featurisation pass with a two-epoch design, or by
  maintaining the counters incrementally in the serving layer (which is what production
  would do anyway, and which also removes the batch-aggregate leakage).
* **`model.predict` over 880 M rows** is ~2 h single-node. Fix: shard the test file and
  run N Kaggle sessions / Dask workers, or convert the model to `treelite`.
* **the catalogue matrices are fine** — 1.3 M articles × 64 is still only 330 MB. Content
  representation is *not* the thing that breaks; the user-side state is.

In [44]:
summary = {
    "articles": N_ART,
    "fit_impressions": beh_fit.height,
    "eval_impressions": beh_eval.height,
    "test_impressions": beh_test.height,
    "embedding": EMB_KIND,
    "best_iteration": int(model.best_iteration or 0),
    "es_auc": float(model.best_score["es"]["auc"]),
    "dev_auc_full": float(np.nanmean(per_group_store["GBDT (all features)"]["auc"])),
    "dev_auc_safe": float(np.nanmean(per_group_store["GBDT (serving-safe)"]["auc"])),
    "recall_bm25_live@100": rec_bm25_pool[100],
    "recall_emb_live@100": rec_emb_pool[100],
}
print(json.dumps(summary, indent=2))
with open(Path(CFG["out_dir"]) / "mind_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

{
  "articles": 130379,
  "fit_impressions": 450000,
  "eval_impressions": 80000,
  "test_impressions": 2370727,
  "embedding": "char-lsa",
  "best_iteration": 42,
  "es_auc": 0.7086706898878781,
  "dev_auc_full": 0.6087450558587885,
  "dev_auc_safe": 0.5961630038632991,
  "recall_bm25_live@100": 0.09116877381877381,
  "recall_emb_live@100": 0.025168779831279832
}
